In [1]:
import json
import uuid
from IPython.display import HTML, display

In [2]:
def visualize_record(record):
    """
    Renders a true Label Studio clone inside Jupyter notebooks.
    Features text auto-wrapping, background color span highlights, 
    and dynamic overhead relation bezier curves.
    """
    text = record.get('text', '')
    needs = record.get('needs', [])
    entities = record.get('entities', [])
    relations = record.get('relations', [])
    
    # 1. Define a vibrant Label Studio color palette
    colors = {
        "PERSON_ROLE": "#7de8ff",
        "PERSON_NAME": "#b388ff",
        "PERSON_PRONOUN": "#ea80fc",
        "HOUSING_CONDITIONS_HOARDING": "#ffadad",
        "CAUTIONS_UNCLEAN_UNSAFE_LIVING_ENVIRONMENT": "#ffd166",
        "PROPERTY_LEVEL_DISREPAIR_DAMP_MOULD": "#98f5e1",
        "HEALTH_MENTAL_HEALTH": "#ffca3a",
        "SAFETY_RISK_DOMESTIC_ABUSE": "#ff595e",
        "CAUTIONS_PHYSICAL_ABUSE_OR_THREAT_OF": "#ff924c"
    }

    # 2. Sort all spans chronologically by character indices
    all_spans = sorted(needs + entities, key=lambda x: x['start'])
    
    html_elements = []
    current_char = 0
    uid = str(uuid.uuid4())[:8] # Unique container hash to isolate multiple runs
    
    # 3. Interweave plain text strings and stylized highlight blocks
    for span in all_spans:
        if span['start'] > current_char:
            html_elements.append(f"<span style='white-space: pre-wrap;'>{text[current_char:span['start']]}</span>")
            
        span_text = text[span['start']:span['end']]
        label_upper = span['label'].upper()
        # Fallback to grey if a new taxonomy category isn't explicit in the palette
        bg_color = colors.get(label_upper, "#e2e8f0") 
        
        # Pull the last section of long names to keep the text footprint tiny (e.g. HOARDING)
        short_label = span['label'].split('_')[-1].upper()
        
        html_elements.append(f"""
            <span id="span-{uid}-{span['id']}" class="ls-pill" style="
                background-color: {bg_color}; 
                padding: 2px 6px; 
                border-radius: 4px; 
                margin: 0 3px; 
                display: inline-block; 
                position: relative; 
                font-weight: 500;
                border: 1px solid rgba(0,0,0,0.15);
                line-height: 1.2;
            ">
                {span_text}
                <span style="
                    display: block; 
                    font-size: 8px; 
                    color: rgba(0,0,0,0.6); 
                    font-weight: 800; 
                    margin-top: 1px; 
                    letter-spacing: 0.5px;
                ">{short_label}</span>
            </span>
        """)
        current_char = span['end']
        
    if current_char < len(text):
        html_elements.append(f"<span style='white-space: pre-wrap;'>{text[current_char:]}</span>")

    # 4. Construct base HTML frame with an absolute overlay vector canvas
    html_output = f"""
    <div id="ls-container-{uid}" style="
        position: relative; 
        line-height: 3.2; 
        padding-top: 75px; 
        padding-bottom: 20px; 
        font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; 
        font-size: 14px;
        color: #1a202c;
        background: #ffffff;
        border-radius: 8px;
        box-shadow: 0 1px 3px rgba(0,0,0,0.1);
        padding-left: 15px;
        padding-right: 15px;
    ">
        <svg id="ls-svg-{uid}" style="
            position: absolute; 
            top: 0; 
            left: 0; 
            width: 100%; 
            height: 100%; 
            pointer-events: none; 
            overflow: visible;
        ">
            <defs>
                <marker id="arrow-{uid}" viewBox="0 0 10 10" refX="6" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
                    <path d="M 0 1.5 L 8 5 L 0 8.5 z" fill="#4a5568"/>
                </marker>
            </defs>
        </svg>
        {"".join(html_elements)}
    </div>
    """

    # 5. Embedded JavaScript: Computes live bounding coordinates and renders paths
    js_script = f"""
    <script>
    (function() {{
        const container = document.getElementById("ls-container-{uid}");
        const svg = document.getElementById("ls-svg-{uid}");
        const relations = {json.dumps(relations)};
        
        // Brief timeout ensures Jupyter cell has rendered elements into DOM
        setTimeout(() => {{
            const containerRect = container.getBoundingClientRect();
            
            relations.forEach((rel, i) => {{
                const fromEl = document.getElementById("span-{uid}-" + rel.from);
                const toEl = document.getElementById("span-{uid}-" + rel.to);
                
                if (fromEl && toEl) {{
                    const fromRect = fromEl.getBoundingClientRect();
                    const toRect = toEl.getBoundingClientRect();
                    
                    // Calculate relative center-points of source and targets
                    const x1 = (fromRect.left + fromRect.width / 2) - containerRect.left;
                    const y1 = fromRect.top - containerRect.top;
                    
                    const x2 = (toRect.left + toRect.width / 2) - containerRect.left;
                    const y2 = toRect.top - containerRect.top;
                    
                    // Calculate curve variance based on distance to prevent line-stacking overlaps
                    const distance = Math.abs(x2 - x1);
                    const arcHeight = Math.min(65, 25 + (distance * 0.12)) + (i * 8);
                    const h = Math.min(y1, y2) - arcHeight;
                    
                    // Draw smooth Quadratic Bezier curve paths
                    const path = document.createElementNS("http://www.w3.org/2000/svg", "path");
                    const d = `M ${{x1}} ${{y1}} Q ${{ (x1 + x2) / 2 }} ${{h}} ${{x2}} ${{y2}}`;
                    
                    path.setAttribute("d", d);
                    path.setAttribute("fill", "none");
                    path.setAttribute("stroke", "#4a5568");
                    path.setAttribute("stroke-width", "1.5");
                    path.setAttribute("marker-end", "url(#arrow-{uid})");
                    svg.appendChild(path);
                    
                    // Center-align the text label along the peak of the curve
                    const text = document.createElementNS("http://www.w3.org/2000/svg", "text");
                    text.setAttribute("x", (x1 + x2) / 2);
                    text.setAttribute("y", h + 12);
                    text.setAttribute("fill", "#4a5568");
                    text.setAttribute("font-size", "8px");
                    text.setAttribute("font-weight", "bold");
                    text.setAttribute("text-anchor", "middle");
                    svg.appendChild(text);
                }}
            }});
        }}, 150);
    }})();
    </script>
    """
    
    display(HTML(html_output + js_script))

In [3]:
with open('../../data/output/generated_fake_data.json', 'r') as f:
    fake_records = json.load(f)

with open('../../data/output/gold_standard.json', 'r') as f:
    records = json.load(f)

In [11]:
n = 110
visualize_record(fake_records[n])
visualize_record(records[n])